# Asymmetric lungs

Batches that sweep `asymmetric_lung_graph` over `flow_ratio`, at one target size or several, with random matches. Other batches go in `../analysis.ipynb`, whose cross-size section needs one graph per type and size.
Reads only the report written by `hpc finalize` (`load` raises if it is missing); nothing here simulates.

Mean times are conditional on fixation. `rho` counts censored runs (hit `max_steps`) as not fixed; the verification report says if there were any.

In [ ]:
%load_ext autoreload
%autoreload 2

import json

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from moran_graphs.plots import COLORS, heatmap, lines, load, print_batch, print_run_report, sweep
from moran_graphs.PopulationGraph import PopulationGraph

The trees at each swept `flow_ratio`, `n_nodes` = 31: trachea on top, one row per generation, major daughter to the left.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(15, 2.5), layout="constrained")
for ax, flow_ratio in zip(axes, [0.1, 0.15, 0.2, 0.25, 0.3, 0.326, 0.35, 0.4, 0.45, 0.5]):
    lung = PopulationGraph.asymmetric_lung_graph(flow_ratio, n_nodes=31).graph
    nx.draw(lung, nx.get_node_attributes(lung, "pos"), ax=ax, node_size=8, width=0.5,
            node_color=COLORS["Asymmetric lung"])
    ax.set_title(f"{flow_ratio} (N = {lung.number_of_nodes()})", fontsize=8)

## Configuration

The only cell to edit.

In [ ]:
BATCHES = ["2026-10-07-asymmetric-lungs-size-sweep"]
R_VALUES = [1.1]  # the per-r figures need exactly one

## What ran

Plan, submission, graphs built, then the verification report of each batch.

In [ ]:
for batch in BATCHES:
    print_batch(batch)

### How the batch went

Your wait from submission to the finished report, CPU time and speed, how much worker time went to waste, and what each r cost.

In [ ]:
for batch in BATCHES:
    print_run_report(batch)

## Load

In [ ]:
results, _ = load(BATCHES)
print(f"{results['hash'].nunique()} graphs, r values {sorted(results['r'].unique().tolist())}")
results = results[results["r"].isin(R_VALUES)]
results.groupby(["category", "r"])[["rho", "steps_mean_given_fixed", "n_censored"]].agg(["count", "mean", "min", "max"])

## Parameter sweep

One category against a kwarg of its recipe, e.g. the asymmetric lungs against `flow_ratio`. Points are colored by N, because `n_nodes` is only a target and the built tree's N moves with the sweep.
Grey bars: the random graphs with the same N and E (min to max), at the same x.
Bottom row: each graph minus the mean of those random graphs (`*_minus_random` columns, added by `load`), which takes N out of the comparison. No error bars there: the spread of the random graphs (grey) is the reference.

Several kwarg values can build the same graph; the duplicates are dropped, so each point is the first value that built its graph (see the `graphs` warning in the verification report).

Dashed line: the human lung, `flow_ratio` = 0.326. In 2026-10-06-asymmetric-lungs it builds the same tree as 0.32, so that point (N = 33) is the human tree.

In [ ]:
SWEEP_CATEGORY, SWEEP_KWARG = "Asymmetric lung", "flow_ratio"
HUMAN_FLOW_RATIO = 0.326  # Kundu & Panchagnula 2023, the asymmetric_lung_graph default

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), layout="constrained", sharex=True)
ys = ("rho", "steps_mean_given_fixed", "rho_minus_random", "steps_mean_given_fixed_minus_random")
for ax, y in zip(axes.flat, ys):
    points = sweep(ax, results, SWEEP_CATEGORY, SWEEP_KWARG, y)
    if y.endswith("_minus_random"):
        ax.axhline(0, color=COLORS["Random"], lw=1)
    ax.axvline(HUMAN_FLOW_RATIO, color="0.2", ls="--", lw=1, label=f"human lung ({HUMAN_FLOW_RATIO})")
fig.colorbar(points, ax=axes, label="N")
axes[0, 0].legend(frameon=False)
fig.suptitle(f"r = {R_VALUES}");

## Asymmetric lungs across sizes

For a batch that sweeps `flow_ratio` at several target sizes (`n_nodes` in the recipe), e.g. 2026-10-07-asymmetric-lungs-size-sweep. One line per `flow_ratio` against the built N, which misses the target by a few percent; grey: the random trees.

- **Left:** fixation probability and mean fixation time (given fixation).
- **Right:** the same divided by the median random tree with the same N and E, as in `versus_random`. Above 1 the lung amplifies more (top) or fixes slower (bottom). Grey here is each random tree over that same median: the spread a lung has to leave to stand out.

Error bars are 95% intervals (Wilson for `rho`, 1.96 SE for time), left only: on the right the reference is the random spread.

In [ ]:
LUNG_CATEGORY = "Asymmetric lung"
HUMAN_FLOW_RATIO = 0.326  # Kundu & Panchagnula 2023

size = ["n_nodes", "n_edges"]
metrics = ["rho", "steps_mean_given_fixed"]
# Every graph, random ones included, over the median random tree of its N and E, as in versus_random.
random_median = results[results["category"] == "Random"].groupby(size)[metrics].median()
matched_median = results[size].join(random_median, on=size)
compared = results.assign(**{f"{y}_over_random": results[y] / matched_median[y] for y in metrics})
lungs = compared[compared["category"] == LUNG_CATEGORY]
recipe = pd.json_normalize(lungs["kwargs"].map(json.loads).tolist()).set_index(lungs.index)
lungs = lungs.assign(flow_ratio=recipe["flow_ratio"], **{"target N": recipe["n_nodes"]})
random = compared[compared["category"] == "Random"]
human = lungs[lungs["flow_ratio"] == HUMAN_FLOW_RATIO].sort_values("n_nodes")

panels = (("rho", "fixation probability", "linear"),
          ("rho_over_random", "fixation probability / median random tree", "linear"),
          ("steps_mean_given_fixed", "mean fixation time (steps), given fixation", "log"),
          ("steps_mean_given_fixed_over_random", "fixation time / median random tree", "log"))
fig, axes = plt.subplots(2, 2, figsize=(14, 10), layout="constrained")
for ax, (y, ylabel, yscale) in zip(axes.flat, panels):
    ax.scatter(random["n_nodes"], random[y], s=8, alpha=0.5, linewidths=0, color=COLORS["Random"],
               label="random trees")
    lines(ax, lungs, "n_nodes", y, by="flow_ratio", cmap="plasma")
    ax.plot(human["n_nodes"], human[y], color="0.1", ls="--", lw=1.5, zorder=4, label="human lung")
    if y.endswith("_over_random"):
        ax.axhline(1, color="0.2", lw=1)
    ax.set(xscale="log", yscale=yscale, ylabel=ylabel)
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="outside right center", frameon=False, fontsize=9)
fig.suptitle(f"Asymmetric lungs across sizes against random trees of the same N and E, r = {R_VALUES}");

### Heatmaps over target N and flow_ratio

The four panels of the line plots above, one cell per (target N, `flow_ratio`).
- **Left:** fixation probability and mean fixation time (given fixation). Time is on a log color scale because it grows by orders of magnitude with N, so these panels mostly show N.
- **Right:** the same over the median random tree with the same N and E, which takes N out: what is left is the effect of `flow_ratio` at each size. Above 1 (red) the lung amplifies more (top) or fixes slower (bottom). Symmetric around 1: linear for fixation probability, log for time (so x2 and x1/2 are equally strong).

Grey: a `flow_ratio` that built the same tree as a smaller one at that size, dropped as a duplicate. Boxed column: the human lung.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9), layout="constrained")
for ax, (y, label, yscale) in zip(axes.flat, panels):
    grid = heatmap(ax, lungs, "flow_ratio", y, "target N", label, yscale)
    human_column = grid.columns.get_loc(HUMAN_FLOW_RATIO)
    ax.add_patch(plt.Rectangle((human_column - 0.5, -0.5), 1, len(grid), fill=False, lw=1.5))
    ax.tick_params(axis="x", labelrotation=45)
fig.suptitle(f"Asymmetric lungs over target N and flow_ratio, r = {R_VALUES}");